# v3 controls (GPU 1): Hindi-only, then Yoruba-only
Same starting model, same number of phases, same phase schedule as the continual run, but the language never changes
(Yoruba-only switches once, from the loaded Hindi model to Yoruba, then repeats Yoruba).
Checkpoints: `runs/v3_only_hi_s0/ckpt/` and `runs/v3_only_yo_s0/ckpt/`. Rerunning resumes each run where it stopped.

In [2]:
# Run this FIRST in a fresh kernel: it pins the notebook to GPU 1 before torch starts.
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from common import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

device: cuda | NVIDIA GeForce RTX 3090


In [3]:
SMOKE = False      # True first, for a quick end-to-end test
MODES = ["only_hi", "only_yo"]
D = load_data(train_langs=["hi", "yo"])

  hi: 993,016 train / 2,000 val
  zh: 2,000 val
  yo: 227,670 train / 2,000 val
data loaded in 22s


In [4]:
for mode in MODES:
    C = RunConfig(run_name=f"v3_{mode}_s0", mode=mode, n_phases=24)
    if SMOKE:   # tiny phases, ~10 minutes, separate run folder
        C.run_name += "_smoke"; C.n_phases = 3; C.eval_every = 50; C.warmup_steps = 50
        C.min_steps = 100; C.cooldown_steps = 100; C.max_stable_steps = 300
    train_run(C, D)

 RUN v3_only_hi_s0 | mode only_hi | 24 phases after phase 0 | device cuda
 phase order: hi (loaded) → hi → hi → hi → hi → hi → hi → hi → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 20,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)
phase 0: evaluating the loaded Hindi model on all languages...

 PHASE 0 DONE | trained: hi | loaded model, no training
--------------------------------------------------------------------------------------
   lang      val       Δ |   chrF      Δ |  chrF (unrestricted)  val (unrestr.)
   hi      1.500         |   45.6        |                 45.6           1.557  ◀ trained
   zh      9.050         |    0.3        |                  0.9          10.565
   yo      8.331         |    1.1        |                  0.6           8.852

▶ PHASE 1/24 | training hi (993,016 pair